In [42]:
import pandas as pd
import os

from neo4j import GraphDatabase
from decimal import Decimal
from dotenv import load_dotenv

load_dotenv("../.env")

NODES_DIR = "../data/nodes/"

class Config:
    def __init__(self, mode="LOCAL"):
        mode = mode.upper()

        if mode == "LOCAL":
            self.URI = os.getenv("NEO4J_URI_LOCAL")
            self.USER = os.getenv("NEO4J_USER_LOCAL")
            self.PASSWORD = os.getenv("NEO4J_PASSWORD_LOCAL")
        elif mode == "GROUP":
            self.URI = os.getenv("NEO4J_GROUP_URI")
            self.USER = os.getenv("NEO4J_GROUP_USER")
            self.PASSWORD = os.getenv("NEO4J_GROUP_PASSWORD")
        else:
            raise ValueError("Mode must be 'LOCAL' or 'GROUP'.")

        self.DATABASE = "neo4j"

config = Config(mode="LOCAL")
driver = GraphDatabase.driver(config.URI, auth=(config.USER, config.PASSWORD))

with driver.session(database=config.DATABASE) as session:
    session.run("MATCH (n) RETURN n LIMIT 1")
    print(f"Connection successful")

def query_neo4j(cypher_query: str, parameters: dict = None):
    with driver.session(database=config.DATABASE) as session:
        result = session.run(cypher_query, parameters)
        return result.data()

Connection successful


In [43]:
def delete_nodes(label: str):
    label_exists = query_neo4j(f"CALL db.labels() YIELD label WHERE label = '{label}' RETURN count(*) > 0 AS exists")[0]['exists']
    if label_exists:
        deleted_count = query_neo4j(f"MATCH (n:`{label}`) DETACH DELETE n RETURN count(n) AS deleted_count")[0]["deleted_count"]
        print(f"Deleted {deleted_count} existing {label} nodes.")
    else:
        print(f"No {label} nodes found.")

def clean_data(data: dict):
    cleaned_data = []
    for row in data:
        cleaned_row = {k: float(v) if isinstance(v, Decimal) else v for k, v in row.items()}
        if cleaned_row:
            cleaned_data.append(cleaned_row)
    return cleaned_data

def create_nodes(label: str, data: dict):
    data = clean_data(data)
    create_query = f"""
        UNWIND $data AS row
        CREATE (n:`{label}`)
        SET n = row
        RETURN count(n) AS created
    """
    result = query_neo4j(create_query, {"data": data})
    created_count = result[0]['created']
    print(f"Created {created_count} new {label} nodes.")

def drop_indexes(label):
    # Step 1: get all indexes for this label
    get_indexes_query = """
        SHOW INDEXES YIELD name, labelsOrTypes
        WHERE $label IN labelsOrTypes
        RETURN name
    """
    results = query_neo4j(get_indexes_query, {"label": label})

    # Step 2: drop each index
    for row in results:
        name = row["name"]
        drop_query = f"DROP INDEX {name}"
        query_neo4j(drop_query)
    print(f"Dropped indexes for label `{label}`.")


def create_index_on_id(label: str):
    index_name = f"{label.lower()}_id_index"
    drop_index_query = f"DROP INDEX {index_name} IF EXISTS"
    create_index_query = f"CREATE INDEX {index_name} FOR (n:`{label}`) ON (n.id)"
    query_neo4j(drop_index_query)
    query_neo4j(create_index_query)
    print(f"Created index `{index_name}` on {label}(id).")

def load_nodes(label, filename: str):
    print(f"\nLoading {label} nodes...")
    df = pd.read_json(NODES_DIR + filename)
    data = df.to_dict(orient="records")
    drop_indexes(label)
    create_index_on_id(label)
    delete_nodes(label)
    create_nodes(label, data)

In [44]:
node_files = os.listdir(NODES_DIR)
labels = {
    "".join(c.capitalize() for c in filename.split(".")[0].split("_")): filename for filename in node_files
}
labels


{'Zipcode': 'zipcode.json',
 'BlockGroup': 'block_group.json',
 'State': 'state.json',
 'Business': 'business.json',
 'City': 'city.json',
 'ZoneType': 'zone_type.json',
 'ZoneLocation': 'zone_location.json',
 'BusinessLocation': 'business_location.json',
 'Community': 'community.json',
 'County': 'county.json'}

In [45]:
for label, file in labels.items():
    load_nodes(label, file)


Loading Zipcode nodes...
Dropped indexes for label `Zipcode`.
Created index `zipcode_id_index` on Zipcode(id).
Deleted 0 existing Zipcode nodes.
Created 105 new Zipcode nodes.

Loading BlockGroup nodes...
Dropped indexes for label `BlockGroup`.
Created index `blockgroup_id_index` on BlockGroup(id).
Deleted 0 existing BlockGroup nodes.
Created 2085 new BlockGroup nodes.

Loading State nodes...
Dropped indexes for label `State`.
Created index `state_id_index` on State(id).
Deleted 0 existing State nodes.
Created 1 new State nodes.

Loading Business nodes...
Dropped indexes for label `Business`.
Created index `business_id_index` on Business(id).
Deleted 0 existing Business nodes.
Created 32010 new Business nodes.

Loading City nodes...
Dropped indexes for label `City`.
Created index `city_id_index` on City(id).
Deleted 0 existing City nodes.
Created 52 new City nodes.

Loading ZoneType nodes...
Dropped indexes for label `ZoneType`.
Created index `zonetype_id_index` on ZoneType(id).
Delet